### Temporal Difference Learning 

`Temporal Difference Learning, Updates the value of a state based on the reward plus the value of the next state.`


TD learning updates value estimates based on other, current estimates, not just waiting for the final outcome. That’s why it’s called "bootstrapping." It learns from incomplete episodes, one step at a time.


Suppose you are learning the value of being in state $s$

- You are in state $s$.
- Take action $a$, get reward $r$, and land in state $s'$
- current $s$ is $V(S)$
- current guess for the value of $s'$ is $V(s')$

The update rule is:

$$
V(S) \leftarrow V(S) + \alpha \left[ R + \gamma V(S') - V(S) \right]
$$

`α` is the learning rate (small step size)  
`γ` is the discount factor



### Random Walk - MC and TD

**Sutton & Barto’s classic 5-state Random Walk** example, define it clearly, then solve it **manually** using both **Monte Carlo (MC)** and **Temporal Difference (TD(0))** methods to see the difference in updates.


**1. Problem Definition – Random Walk**

We have **7 states** in a line:

```
Terminal:      A     B     C     D     E     F     Terminal
              (1)   (2)   (3)   (4)   (5)   (6)
```

* States **A–F** are numbered `1` to `6`.
* State `0` (left terminal) ends the episode with **reward = 0**.
* State `7` (right terminal) ends the episode with **reward = 1**.
* The agent starts in the **middle**: `C` (state 3).
* At each step, the agent moves **Left** or **Right** with equal probability (0.5).
* Goal: Estimate the **state value function** $V(s)$ = expected return starting from `s` until termination.

**True values** (for reference) in Sutton & Barto’s example are:

$$
V(A)=\frac{1}{6},\ V(B)=\frac{2}{6},\ V(C)=\frac{3}{6},\ V(D)=\frac{4}{6},\ V(E)=\frac{5}{6},\ V(F)=\frac{6}{6}
$$

(but we’ll try to learn them from experience).

---

**2. Manual Example Episode**

Let’s simulate **one episode** from C (state 3):

Suppose the random moves go like this:

```
C → D → E → F → Terminal (reward 1)
```

Rewards along the way:

* From F to Terminal: $R = 1$
* All other moves: $R = 0$

So the sequence of `(state, reward)` is:

| Time | State    | Reward from this step |
| ---- | -------- | --------------------- |
| 0    | C        | 0                     |
| 1    | D        | 0                     |
| 2    | E        | 0                     |
| 3    | F        | 1                     |
| 4    | Terminal | —                     |

---

**3. Monte Carlo Update (First-Visit MC, α = 0.1)**

In MC, we wait until the episode ends, then **for each state visited, update V(s) toward the *actual return*** from that point.

Step 1: Compute returns

From the above episode:

* $G(C) = 1$ (eventually you reach terminal with reward 1)
* $G(D) = 1$
* $G(E) = 1$
* $G(F) = 1$

Step 2: Update V(s)

Let’s assume we start with $V(s) = 0.5$ for all non-terminal states.

For each state $s$ visited once in episode:

$$
V(s) \leftarrow V(s) + \alpha [G(s) - V(s)]
$$

with α = 0.1:

* $V(C) = 0.5 + 0.1 (1 - 0.5) = 0.5 + 0.05 = 0.55$
* $V(D) = 0.5 + 0.1 (1 - 0.5) = 0.55$
* $V(E) = 0.5 + 0.1 (1 - 0.5) = 0.55$
* $V(F) = 0.5 + 0.1 (1 - 0.5) = 0.55$

All others remain at 0.5.

---

**4. TD(0) Update (α = 0.1, γ = 1)**

In TD(0), we update **after every step**:

$$
V(s) \leftarrow V(s) + \alpha [R + \gamma V(s') - V(s)]
$$

Initial values: all 0.5.

Step-by-step:

1. **From C to D**, R=0:
   $V(C) = 0.5 + 0.1[0 + 1 \times 0.5 - 0.5] = 0.5$ (no change)
2. **From D to E**, R=0:
   $V(D) = 0.5 + 0.1[0 + 0.5 - 0.5] = 0.5$ (no change)
3. **From E to F**, R=0:
   $V(E) = 0.5 + 0.1[0 + 0.5 - 0.5] = 0.5$ (no change)
4. **From F to Terminal**, R=1, V(Terminal)=0:
   $V(F) = 0.5 + 0.1[1 + 0 - 0.5] = 0.5 + 0.05 = 0.55$

So after one episode:

* TD updated **only F** (because the others saw no immediate value difference in this episode).
* MC updated **all visited states** after seeing the full return.

---

**5. Key Contrast**

| Feature               | Monte Carlo                    | TD(0)                                             |
| --------------------- | ------------------------------ | ------------------------------------------------- |
| Update Timing         | After episode ends             | After each step                                   |
| Uses Full Return?     | Yes                            | No – uses current estimate (bootstraps)           |
| This Episode’s Effect | All visited states get updated | Only states with a nonzero TD error in their step |
| Variance              | Higher (waits for full return) | Lower (uses estimates early)                      |



In [56]:
#TD policy evaluation

import random 
from collections import defaultdict
from typing import Callable, Dict, Tuple, List, Optional

In [57]:
# creating a Random Walk Env 

class RandomWalk1d:
    """
    Sutton & Barto style random walk:
        States: 0 (terminal, reward=0), 1..N (non-terminal), N+1 (terminal, reward=1)
        Start in middle: s0 = (N+1)//2
        Actions: -1 (left), +1 (right)
        Dynamics: deterministic given action (keeps within [0, N+1])
    """

    def __init__(self, N: int = 5, seed:int = 42):
        """
        Args:
            N: number of non-terminal states (classically 5). Terminals are 0 and N+1.
        """
        assert N >= 2, "Requires at leat 2 terminla states"
        self.N = N 
        self.left_terminal = 0
        self.right_terminal = N+1
        self.state_space = list(range(self.left_terminal, self.right_terminal+1))
        self.non_terminal_state = list(range(1, self.right_terminal))

        self.actions = [-1, +1]

        # always starts at middle
        self.start_state = self.right_terminal // 2 
        self.rng = random.Random(seed)

        # reset env 
        self.reset()
    
    def reset(self)->int:
        "reset epiosde to start"
        self.s = self.start_state
        return self.s

    def step(self, action:int)->Tuple[int, float, bool]:
        """
        Take an action (-1 for left, +1 for right).
        Returns: (next_state, reward, done)
        """

        assert action in [-1, +1], "Invalid action, must be left or right [-1, +1]"
        s_next = self.s + action # moves to left or right 
        s_next = max(self.left_terminal, min(self.right_terminal, s_next)) #assuring we are within state_space

        # we only have rewards at the terminal states 
        r = 1.0 if s_next == self.right_terminal else 0.0

        # check if we end of episode by checnking if we reached terminal state 
        done = s_next in (self.left_terminal, self.right_terminal)
        self.s = s_next
        return s_next, r, done


In [58]:
# lets define the policy to eval 
def equal_random_policy()->int:
    return random.choice([-1, +1])

def always_right_policy()->int:
    return +1


# evaluate the policy with TD(0)
def td0_policy_eval(env:RandomWalk1d, policy:Callable, alpha:float = 0.1, gamma:float= 1.0, num_eps:int = 200, init_value: float = 0.5)->Dict[int, float]:
    """
    Evaluate a given policy with TD(0) on state values V(s).

    Args:
        env: environment with reset() and step(a) -> (s', r, done)
        policy: mapping state -> action (deterministic). For stochastic, sample inside.
        alpha: step-size
        gamma: discount factor
        num_eps: how many episodes to run
        init_value: initial V(s) for non-terminals

    Returns:
        V: dict mapping state -> value estimate. Terminal states fixed at their true values.
    """
    V = defaultdict(lambda: init_value)
    V[env.left_terminal] = 0.0
    V[env.right_terminal] = 1.0

    for _ in range(num_eps):
        s = env.reset()
        done = False

        while not done: 
            a = policy()

            s_next, r, done = env.step(a)
            
            # TD(0) update only for non-terminal s
            if s not in (env.left_terminal, env.right_terminal):
                td_target = r + gamma * V[s_next]
                td_error = td_target - V[s]
                # print(f"Updating: V[{s}]:{V[s]} by {alpha * td_error} ")
                # update the current state value 
                V[s] += alpha * td_error
                
            s = s_next 
    return dict(V)


In [59]:
env = RandomWalk1d(N=5, seed=123)

for p in [equal_random_policy, always_right_policy]:
    print(f"Computed state value with {p.__name__} policy")
    V = td0_policy_eval(env=env, policy=p, num_eps = 5000)

    # print 
    states = list(range(env.left_terminal, env.right_terminal + 1))
    for s in states:
        tag = "(T)" if s in (env.left_terminal, env.right_terminal) else "   "
        print(f"s={s}: V={V.get(s, 0.0):.3f} {tag}")



Computed state value with equal_random_policy policy
s=0: V=0.000 (T)
s=1: V=0.348    
s=2: V=0.570    
s=3: V=0.954    
s=4: V=1.316    
s=5: V=1.796    
s=6: V=1.000 (T)
Computed state value with always_right_policy policy
s=0: V=0.000 (T)
s=1: V=0.000    
s=2: V=0.000    
s=3: V=2.000    
s=4: V=2.000    
s=5: V=2.000    
s=6: V=1.000 (T)


Temporal Difference (TD) prediction (learning $V(s)$ ), let’s move to TD control, where the goal is to learn action-values $Q(s,a)$ and find an optimal policy.



**1. What is Temporal Difference Control?**

In **TD prediction**, we learned $V(s)$ for a *given* policy.
In **TD control**, we:

1. **Learn $Q(s,a)$** — the expected return starting from state $s$, taking action $a$, and following the policy thereafter.
2. **Improve the policy** towards the optimal one ($\pi^*$) as we learn.

**Why?**
Knowing $Q(s,a)$ allows us to pick the best action in each state — solving the RL problem.

---

**2. Key TD Control Algorithms**

The two most common are:

| Algorithm      | On/Off Policy | Update Target                    |
| -------------- | ------------- | -------------------------------- |
| **SARSA**      | On-policy     | $R + \gamma Q(s', a')$           |
| **Q-learning** | Off-policy    | $R + \gamma \max_{a'} Q(s', a')$ |


**2.1 SARSA (On-Policy TD Control)**

Named after the sequence:

$$
(s, a, r, s', a')
$$

Update rule:

$$
Q(s,a) \leftarrow Q(s,a) + \alpha \big[ r + \gamma Q(s',a') - Q(s,a) \big]
$$

* Learns the value of the policy you’re *currently using* (including exploration steps).
* Good for tasks where following the exploration strategy’s consequences is important (e.g., cliff walking).


**2.2 Q-Learning (Off-Policy TD Control)**

Update rule:

$$
Q(s,a) \leftarrow Q(s,a) + \alpha \big[ r + \gamma \max_{a'} Q(s',a') - Q(s,a) \big]
$$

* Learns the value of the **greedy** policy while behaving with exploration.
* Tends to be more aggressive and can converge faster to optimal $Q^*$ in many cases.

---

## **3. Manual Example (Small MDP)**

Let’s use a **2-state MDP**:

States: S1, S2; Actions: Left, Right
Transition & reward:

* From S1:

  * Left → S2, reward 0
  * Right → Terminal, reward 1
* From S2:

  * Left → Terminal, reward 0
  * Right → S1, reward 0

Start $Q(s,a) = 0$, α = 0.5, γ = 1.

---

**Episode (SARSA)**

Suppose:

1. Start S1, choose Left → S2, reward 0
2. From S2, choose Right → S1, reward 0
3. From S1, choose Right → Terminal, reward 1

**Updates:**

* Step 1: S1, Left, reward 0, S2, action Right:

  $$
  Q(S1,L) = 0 + 0.5[0 + 1 \times Q(S2,R) - 0] = 0
  $$

  (no change yet, since Q(S2,R) is 0)

* Step 2: S2, Right, reward 0, S1, action Right:

  $$
  Q(S2,R) = 0 + 0.5[0 + 1 \times Q(S1,R) - 0] = 0
  $$

  (no change yet)

* Step 3: S1, Right, reward 1, Terminal:

  $$
  Q(S1,R) = 0 + 0.5[1 + 0 - 0] = 0.5
  $$

So after one episode:

$$
Q(S1,R) = 0.5,\quad Q(S1,L) = 0,\quad Q(S2,R) = 0
$$

SARSA updates step-by-step using **the action actually taken next**.

---

**Episode (Q-Learning)**

Same experience, but updates use the **max over next actions**:

* Step 1: S1, Left → S2, reward 0:

  $$
  Q(S1,L) = 0 + 0.5[0 + \max(Q(S2,L), Q(S2,R)) - 0] = 0
  $$
* Step 2: S2, Right → S1, reward 0:

  $$
  Q(S2,R) = 0 + 0.5[0 + \max(Q(S1,L), Q(S1,R)) - 0] = 0
  $$
* Step 3: S1, Right → Terminal, reward 1:

  $$
  Q(S1,R) = 0 + 0.5[1 + 0 - 0] = 0.5
  $$

Here, Q-learning **learns as if it always took the best action next**, even if in the episode it didn’t.

---

## **4. Why TD Control Works**

* It **bootstraps** from future estimates, so learning is faster than Monte Carlo control.
* It can learn **online** and **incomplete episodes**.
* With enough exploration and small α, it converges to optimal $Q^*$.


In [71]:
#we will use the same environment Random walk for TD control using SARSA, and Q- learning. 

# epsilon greedy 
def epsilon_greedy(Q:dict, state: int, actions: List[int], epsilon:float)->int:
    """Select action ε-greedily from Q-values."""
    if random.random() < epsilon: return random.choice(actions)
    else:
        # choose the action with max-Q-value
        q_vals = [Q[(state, a)] for a in actions]
        max_q = max(q_vals)
        # tie breaking with random
        best_actions = [a for a, q in zip(actions, q_vals) if q == max_q]
        return random.choice(best_actions)

# print learned policy 
def print_policy(Q, actions, N, title):
    print(f"{title} (epsilon-greedy policy)")
    for s in range(1, N+1):
        q_vals = {a: Q[(s,a)] for a in actions}
        best_action = max(q_vals, key=q_vals.get)
        action_symbol = "->" if best_action == +1 else "<-"
        print(f"State:{s}: {action_symbol} Q:{q_vals}")

In [68]:
#SARSA algorithm 
def sarsa(env:RandomWalk1d, num_eps:int = 2000, alpha:float = 0.1, gamma:float = 1.0, epsilon:float=0.1):
    Q = defaultdict(float)
    # run episods
    for _ in range(num_eps):
        s = env.reset()
        a = epsilon_greedy(Q, s, env.actions, epsilon)
        done = False
        while not done:
            s_next, r, done = env.step(a)
            if not done:
                a_next = epsilon_greedy(Q, s_next, env.actions, epsilon)
                target = r + gamma * Q[(s_next, a_next)]
            else:
                target = r 
            
            # update the action value 
            Q[(s,a)] += alpha * (target - Q[(s,a)])
            s = s_next 
            a = a_next if not done else None
    return Q


# Q-Learning 
def q_learning(env:RandomWalk1d, num_eps:int = 2000, alpha:float = 0.1, gamma:float = 1.0, epsilon:float= 0.1):
    Q = defaultdict(float)
    # run episodes 
    for _ in range(num_eps):
        s = env.reset()
        done = False 
        
        while not done:
            a = epsilon_greedy(Q, s, env.actions, epsilon)
            s_next, r, done = env.step(a)

            if not done:
                max_next_q = max(Q[(s_next, a_next)] for a_next in env.actions)
                target = r + gamma * max_next_q
            else:
                target = r
            
            # update the action-value 
            Q[(s, a)] += alpha * (target - Q[(s,a)])
            s = s_next
    return Q


In [74]:
env = RandomWalk1d(N=5)
Q_qlearn = q_learning(env=env, num_eps = 1000, epsilon = 0.4)
Q_sarsa = sarsa(env=env, num_eps = 1000, epsilon = 0.4)

print_policy(Q_qlearn, env.actions, env.N, "Q-Learning")
print_policy(Q_sarsa, env.actions, env.N, "SARSA")

Q-Learning (epsilon-greedy policy)
State:1: -> Q:{-1: 0.0, 1: 0.9948088571474321}
State:2: -> Q:{-1: 0.9789594517490666, 1: 0.9999999999997997}
State:3: -> Q:{-1: 0.9999999999964053, 1: 0.9999999999999987}
State:4: -> Q:{-1: 0.9999999999999926, 1: 0.9999999999999991}
State:5: -> Q:{-1: 0.9999999999977252, 1: 0.9999999999999996}
SARSA (epsilon-greedy policy)
State:1: -> Q:{-1: 0.0, 1: 0.9263314330901236}
State:2: -> Q:{-1: 0.8420226211241185, 1: 0.9746489299234201}
State:3: -> Q:{-1: 0.933439627849552, 1: 0.9930751830800131}
State:4: -> Q:{-1: 0.9592261333184162, 1: 0.9988391374915889}
State:5: -> Q:{-1: 0.9949539655030928, 1: 0.9999999999999996}


Lets develop cliff walking environment and find optimal policy with TD control - SARSA, and Q-Learning 

**1. Environment**
- Grid (4X12), Bottom left (3,0): start, bottom right (3, 11) goal
- Cliff cells: (3,1) .. (3,10)
- Actions: left, right, up, and down
- Transitions: Move 1 cell, if bump wall stay in place
- Rewards:
    - Step on safe cell: -1
    - Step in cliff: -100 and teleport to start (episode continues)
    - Reaching goal: episode terminates (the -1 is still applied for that move if you step into goal)
- Discount $\gamma$ = 1.0 

**2. Value Representation**
- Learn action-values $Q(s,a)$ in table/dict keyed by (row, col, action)

**3. Behavior policy (exploration)**
- $\epsilon$-greedy over current $Q(s, .)$
- Tie breaks randomly between best actions 

**$. TD-control updates**
- SARSA: uses the actual next action $a'$ chosen by ε-greedy
$$
Q(s,a) \leftarrow Q(s,a) + \alpha \big[ r + \gamma Q(s',a') - Q(s,a) \big]
$$

- Q-Learning: uses the greedy next-action value at $s'$
$$
Q(s,a) \leftarrow Q(s,a) + \alpha \big[ r + \gamma \max_{a'} Q(s',a') - Q(s,a) \big]
$$

In [122]:
class CliffWorld:
    """
    4x12 grid; bottom row is y=3 (0-indexed).
    Start S=(3,0), Goal G=(3,11), Cliff={(3,1)...(3,10)}.
    Reward: -1 per step; stepping into cliff => -100 and teleport to S; episode continues.
    Episode ends when reaching G.
    """
    def __init__(self, rows:int = 4, cols:int = 12, seed:int = 42):
        self.rows = rows
        self.cols = cols

        self.start = (rows-1, 0) #(3,0)
        self.goal = (rows-1, cols-1) # (3,11)
        self.cliff = {(rows-1, col) for col in range(1, cols-1)} # all bottom rows except start and goal state 

        self.ACTIONS = {
            0: (-1, 0),  #UP
            1: (1, 0), # down
            2: (0, -1), #left
            3: (0, 1) # right
        }

        self.ACTION_SYMBOL = {0:'↑', 1:'↓', 2:'←', 3:'→'}
        self.reset()

    def reset(self)->Tuple[int, int]:
        self.s = self.start
        return self.s 

    def step(self, a: int)->Tuple[Tuple[int, int], float, bool]:
        """Apply action; return (next_state, reward, done)."""
        assert a in self.ACTIONS,  "Unknown action"
        row, col = self.s
        action_row, action_col = self.ACTIONS[a]
        next_row, next_col = row + action_row, col + action_col 

        # clip to the grid bounds 
        next_row = max(0, min(self.rows -1, next_row))
        next_col = max(0, min(self.cols -1, next_col))

        # construct next state
        next_state = (next_row, next_col)

        # default step cost 
        reward = -1.0
        done = False 
        
        if next_state in self.cliff:
            reward = -100.0
            next_start = self.start 
            done = False 
        # Reaching goal ends episode (keep -1 step cost as defined)
        elif next_state == self.goal: done = True 
        else: self.s = next_state 

        return next_state, reward, done

In [132]:
def eps_greedy(Q: Dict[Tuple[Tuple[int,int], int], float], s: Tuple[int, int], actions:List[int], epsilon:float)->int:
    random.Random(42)
    actions = list(actions)
    if random.random() < epsilon:
        return random.choice(actions)
    else:
        q_vals = [Q[(s, a)] for a in actions]
        max_q = max(q_vals)
        best_actions = [a for a, q_val in zip(actions, q_vals) if q_val == max_q]
        return random.choice(best_actions)

#Utilities to view learned policy
def greedy_policy_grid(env: CliffWorld, Q: Dict[Tuple[Tuple[int,int], int], float]) -> List[List[str]]:
    grid = [["." for _ in range(env.cols)] for _ in range(env.rows)]
    for r in range(env.rows):
        for c in range(env.cols):
            s = (r, c)
            if s == env.start: grid[r][c] = "S"; continue
            if s == env.goal:  grid[r][c] = "G"; continue
            if s in env.cliff: grid[r][c] = "C"; continue
            # choose greedy action symbol
            best_a = max(env.ACTIONS.keys(), key=lambda a: Q[(s, a)])
            grid[r][c] = env.ACTION_SYMBOL[best_a]
    return grid

def print_grid(grid: List[List[str]]):
    for row in grid:
        print(" ".join(row))

In [133]:
def sarsa(env:CliffWorld, episodes:int = 100, alpha:float = 0.5, gamma:float = 1.0, epsilon:float = 0.1, seed:int = 42):
    Q = defaultdict(float)
    returns = []

    for ep in range(episodes):
        s = env.reset()
        done = False 
        G = 0.0 

        while not done:
            action = eps_greedy(Q=Q, s=s, actions =env.ACTIONS.keys(), epsilon=epsilon)
            s_next, r, done = env.step(action)
            
            # expected returns (since gamma is 1)
            G += r 

            if not done:
                next_action = eps_greedy(Q=Q, s=s_next,  actions =env.ACTIONS.keys(), epsilon=epsilon)
                target = r + gamma * Q[(s_next, next_action)]
            else:
                target = r 
            
            # update the Q value 
            Q[(s, action)] += alpha * (target - Q[(s, a)])
            s = s_next
        # append expected returns 
        returns.append(G)
    return Q, returns 

def q_learning(env:CliffWorld, episodes:int = 100, alpha:float = 0.5, gamma:float = 1.0, epsilon:float = 0.1, seed:int = 42):
    Q = defaultdict(float)
    returns = []

    for eps in range(episodes):
        s = env.reset()
        done = False 
        G = 0.0 

        while not done:
            action = eps_greedy(Q=Q, s=s, actions =env.ACTIONS.keys(), epsilon=epsilon)
            s_next, r, done = env.step(action)
            G += r 

            max_next = max(Q[(s_next, a)] for a in env.ACTIONS.keys()) if not done else 0.0

            target = r + gamma * max_next
            Q[(s,action)] += alpha * (target - Q[(s,action)])

            s = s_next
        returns.append(G)
    
    return Q, returns 


In [134]:
env = CliffWorld()
alpha = 0.5
gamma = 1.0
epsilon = 0.1
episodes = 8000

Q_sarsa, ret_sarsa = sarsa(env, episodes, alpha, gamma, epsilon)
Q_ql,    ret_ql    = q_learning(env, episodes, alpha, gamma, epsilon)

print("\nGreedy policy learned by SARSA (safer detour expected):")
print_grid(greedy_policy_grid(env, Q_sarsa))

print("\nGreedy policy learned by Q-learning (cliff-hugging path expected):")
print_grid(greedy_policy_grid(env, Q_ql))

# Show average return in last 100 episodes (rough performance proxy)
avg_last_100_sarsa = sum(ret_sarsa[-100:]) / 100
avg_last_100_ql    = sum(ret_ql[-100:]) / 100
print(f"\nAvg return (last 100 eps): SARSA = {avg_last_100_sarsa:.2f} | Q-learning = {avg_last_100_ql:.2f}")


Greedy policy learned by SARSA (safer detour expected):
→ → → → → → → → → ↑ → ↓
→ → → → ↓ → → → → → → ↓
→ ↑ ↓ ↓ → ↓ → → → → → ↓
S C C C C C C C C C C G

Greedy policy learned by Q-learning (cliff-hugging path expected):
↓ ↓ → → ↓ → ↓ → → → → ↓
↓ ↓ ↓ ↓ ↓ ↓ ↓ ↓ ↓ ↓ ↓ ↓
→ → → → → → → → → → → ↓
S C C C C C C C C C C G

Avg return (last 100 eps): SARSA = -145.82 | Q-learning = -39.35


(0, 0)